In [ ]:
# Print timestamp for tracking when notebook was run
# Useful for experiment tracking and reproducibility
import datetime
print(f"Notebook last run (end-to-end): {datetime.datetime.now()}")

In [ ]:
# Check GPU availability and memory using nvidia-smi
# This helps verify GPU is accessible for training
!nvidia-smi

In [ ]:
# Import necessary libraries
import os
import tensorflow as tf

In [ ]:
# Display TensorFlow version to ensure compatibility
# Different versions may have different API behaviors
print("Tensorflow version:")
print(tf.__version__ + "\n")

### Download helper functions

In [ ]:
# Download utility functions from external repository
# This avoids duplicating code and provides tested helper functions
import requests

url = 'https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py'
r = requests.get(url)
# Write the downloaded content to a local Python file
with open('helper_functions.py', 'w') as f:
    f.write(r.text)

In [ ]:
# Import helper functions for:
# - create_tensorboard_callback: Track training metrics
# - plot_loss_curves: Visualize training history
# - unzip_data: Extract compressed datasets
# - walk_through_dir: Explore directory structure
from helper_functions import create_tensorboard_callback, plot_loss_curves, unzip_data, walk_through_dir


### Downloading the data

In [ ]:
import urllib.request
import zipfile
import os

# Download the Food-101 subset dataset (10 classes, varying data percentages)
# This dataset contains images of different food categories
# Available versions: 1%, 10%, or 100% of data

# Uncomment the dataset version you want to use:
# extract_dir = "10_food_classes_1_percent"   # Smallest dataset for quick experiments
# extract_dir = "10_food_classes_10_percent"  # Medium dataset for balanced training
extract_dir = "10_food_classes_all_data"     # Full dataset for best performance

# Construct download URL based on selected dataset
url = "https://storage.googleapis.com/ztm_tf_course/food_vision/{}.zip".format(extract_dir)
zip_path = "{}.zip".format(extract_dir)

# Check if dataset already exists to avoid re-downloading
if not os.path.exists(extract_dir):
    print(f"Downloading {url}...")
    
    # Download the zip file from the URL
    urllib.request.urlretrieve(url, zip_path)
    print(f"Downloaded to {zip_path}")
    
    # Extract all files from the zip archive
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    
    # Remove the zip file to save disk space
    os.remove(zip_path)
    print(f"Extracted to {extract_dir}/ and removed {zip_path}")
else:
    print(f"Dataset already exists at {extract_dir}/")

#### 10% of data set 

In [ ]:
# Load 10% dataset using image_dataset_from_directory
# This is a more modern alternative to ImageDataGenerator

# Define paths to training and test directories
train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test/"

# Standard input size for EfficientNet models
IMG_SIZE = (224, 224)

# Create training dataset
# image_dataset_from_directory automatically:
# - Infers labels from subdirectory names
# - Returns a tf.data.Dataset (more efficient than ImageDataGenerator)
train_data_10 = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size = IMG_SIZE,            # Resize all images to 224x224
    label_mode = "categorical",       # One-hot encode labels for multi-class classification
    batch_size = 32                   # Number of images per batch
)

# Create test dataset with same parameters
# Note: batch_size defaults to 32 if not specified
test_data_10 = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size = IMG_SIZE,
    label_mode = "categorical"
)

# Verify class names extracted from directory structure
train_data_10.class_names

### Setting up callbacks

In [ ]:
# Create a function to generate TensorBoard callbacks for experiment tracking
# TensorBoard allows visualization of training metrics, model graphs, and more
# Callback will be saved in folders according to following schema:
# [dir_name]/[experiment_name]/[current_timestamp]

def create_tensorboard_callback(dir_name, experiment_name):
    """
    Creates a TensorBoard callback for tracking training progress.
    
    Args:
        dir_name: Root directory for logs
        experiment_name: Name of the experiment (e.g., model architecture)
    
    Returns:
        TensorBoard callback configured with timestamped log directory
    """
    # Create hierarchical log directory with timestamp for unique identification
    log_dir = dir_name + "/" + experiment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    
    # Initialize TensorBoard callback
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir = log_dir
    )

    print(f"Saving TensorBoard log files to: {log_dir}")
    return tensorboard_callback

### Model 0: a transfer learning model using the Keras Functional API


In [ ]:
# ============ Model 0: Basic Transfer Learning with Functional API ============
# This model demonstrates the Functional API approach (vs Sequential)

# 1. Create base model using EfficientNetV2B0
# EfficientNetV2 is faster and more accurate than V1, optimized for training speed
base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False)

# 2. Freeze the base model to preserve pre-learned patterns from ImageNet
# This is the "transfer" in transfer learning - we keep the learned features
base_model.trainable = False

# 3. Create input layer with explicit shape
# Functional API requires explicit input definition
inputs = tf.keras.layers.Input(shape=(224,224,3), name="input_layer")

# 5. Pass inputs through the base model
# Note: EfficientNetV2 has built-in preprocessing, so no manual normalization needed
x = base_model(inputs)

# Check the output shape after base model
# Shape will be (None, height, width, channels) where height/width depend on pooling
print(f"Shape after base_model: {x.shape}")

# 6. Add global average pooling to convert feature maps to a single vector
# This reduces spatial dimensions (H x W) to a single value per channel
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
print(f"After GlobalAveragePooling2D(): {x.shape}")

# 7. Add output layer with 10 classes (one for each food category)
# Softmax converts logits to probabilities that sum to 1
outputs = tf.keras.layers.Dense(10, activation="softmax", name="output_layer")(x)

# 8. Create the model by connecting inputs to outputs
model_0 = tf.keras.Model(inputs, outputs)

# 9. Compile with appropriate loss and optimizer
model_0.compile(
    loss="categorical_crossentropy",      # For multi-class with one-hot labels
    optimizer = tf.keras.optimizers.Adam(), # Adaptive learning rate optimizer
    metrics=["accuracy"]                   # Track accuracy during training
)

In [ ]:
# Train Model 0 on 10% of the data
history_model_0 =  model_0.fit(
    train_data_10,
    epochs = 5,                                        # Number of complete passes through training data
    steps_per_epoch = len(train_data_10),             # Number of batches per epoch
    validation_data = test_data_10,                   # Data for validation metrics
    validation_steps = int(0.25*len(test_data_10)),   # Only validate on 25% of test data (faster)
    callbacks=[create_tensorboard_callback("transfer_learning", "model_0_data_10_percent")]
)

In [ ]:
# Optional: View base model architecture
# Uncomment to see the detailed layer structure of EfficientNetV2B0
# base_model.summary()

In [ ]:
# Display the complete model architecture
# Shows input layer -> base_model -> pooling -> output
model_0.summary()
     

In [ ]:
# Visualize training progress
# This will show loss and accuracy curves for both training and validation
plot_loss_curves(history_model_0)

#### 1% of dataset

In [ ]:
# Load the smallest dataset (1% of full data)
# This is useful for quick experiments and testing data augmentation effectiveness

# Define paths to 1% dataset
train_dir = "10_food_classes_1_percent/train/"
test_dir = "10_food_classes_1_percent/test/"

IMG_SIZE = (224, 224)

# Create training dataset from 1% data
train_data_1 = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size = IMG_SIZE,
    label_mode = "categorical",
    batch_size = 32
)

# Create test dataset from 1% data
test_data_1 = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size = IMG_SIZE,
    label_mode = "categorical"
)

# Verify same class names as larger datasets
train_data_1.class_names

###  data augmentation 

In [ ]:
# Import Keras layers for building data augmentation pipeline
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# Create a data augmentation pipeline as a Sequential model
# Data augmentation artificially increases dataset size and helps prevent overfitting
# by creating variations of training images

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),    # Randomly flip images left-right (food looks same)
    layers.RandomRotation(0.2),         # Randomly rotate ±20% * 360° = ±72°
    layers.RandomZoom(0.2),             # Randomly zoom in/out by up to 20%
    layers.RandomHeight(0.2),           # Randomly adjust height by up to 20%
    layers.RandomWidth(0.2)             # Randomly adjust width by up to 20%
], name="data_augmentation_01")

In [ ]:
# Import libraries for visualizing augmented images
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random

In [ ]:
# Visualize the effect of data augmentation on a random image
# This helps verify augmentations are reasonable and don't distort images too much

# Select a random class from the dataset
target_class = random.choice(train_data_1.class_names)
# Build path to that class's directory
target_dir = "10_food_classes_1_percent/train/" + target_class
# Choose a random image from that class
random_image = random.choice(os.listdir(target_dir))
# Create full path to the chosen image
random_image_path = target_dir + "/" + random_image

# Load and display the original image
img = mpimg.imread(random_image_path)
plt.imshow(img)
plt.title(f"Original random image from class: {target_class}")
plt.axis(False)  # Hide axis for cleaner visualization

# Apply augmentation and display result
# expand_dims adds batch dimension: (H, W, 3) -> (1, H, W, 3)
# Data augmentation expects batch dimension
augmented_img = data_augmentation(tf.expand_dims(img, axis=0))
plt.figure()
# squeeze removes batch dimension, divide by 255 to normalize to [0,1]
plt.imshow(tf.squeeze(augmented_img)/255.)
plt.title(f"Augmented random image from class: {target_class}")
plt.axis(False)

# Print shapes to understand transformations
print( np.shape(img) )           # Original: (H, W, 3)
print( np.shape(augmented_img) ) # Augmented: (1, H, W, 3)

### Model 1: feature extraction transfer learning model on 1% of the data with data augmentation

In [ ]:
# Redefine data augmentation pipeline for Model 1
# (same as before - could reuse the previous definition)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomHeight(0.2),
    layers.RandomWidth(0.2)
], name="data_augmentation_01")

In [ ]:
# ============ Model 1: Transfer Learning with Data Augmentation (1% data) ============
# Key difference from Model 0: adds data augmentation layer before base model

input_shape = (224,224,3)

# Load pre-trained EfficientNetV2B0 without top classification layer
base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False)
# Freeze base model - we're only training the new layers we add
base_model.trainable = False

# Create input layer
inputs = tf.keras.layers.Input(shape=input_shape, name="input_layer")

# Apply data augmentation to input
# Augmentation only happens during training, not inference
x = data_augmentation(inputs)

# Pass augmented images through base model
# training=False ensures BatchNormalization layers use inference mode
x = base_model(x, training=False)

# Pool feature maps to single vector per image
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
print(f"After GlobalAveragePooling2D(): {x.shape}")

# Add classification layer for 10 food classes
outputs = tf.keras.layers.Dense(10, activation="softmax", name="output_layer")(x)

# Build and compile the model
model_1 = tf.keras.Model(inputs, outputs)
model_1.compile(
    loss="categorical_crossentropy",
    optimizer = tf.keras.optimizers.Adam(),
    metrics=["accuracy"]
)

In [ ]:
# Train Model 1 on 1% of data with augmentation
# This tests if augmentation can compensate for limited training data
history_model_1 =  model_1.fit(
    train_data_1,                                     # Only 1% of full dataset
    epochs = 5,
    steps_per_epoch = len(train_data_1),
    validation_data = test_data_1,
    validation_steps = int(0.25*len(test_data_1)),   # Validate on 25% of test data
    callbacks=[create_tensorboard_callback("transfer_learning", "model_1_data_1_percent")]
)

In [ ]:
# View model architecture
# Note the data_augmentation layer before the base model
model_1.summary()

In [ ]:
# Visualize training curves with augmentation on 1% data
# Compare these curves to Model 0 to see augmentation's effect
plot_loss_curves(history_model_1)
     

### Model 2: feature extraction transfer learning model on 10% of the data with data augmentation


In [ ]:
# ============ Model 2: Transfer Learning with Data Augmentation (10% data) ============
# Same architecture as Model 1, but trained on 10x more data
# This establishes a baseline before fine-tuning

input_shape = (224,224,3)

base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False)
base_model.trainable = False  # Feature extraction only, no fine-tuning yet

inputs = tf.keras.layers.Input(shape=input_shape, name="input_layer")
x = data_augmentation(inputs)  # Apply augmentation
x = base_model(x, training=False)  # Extract features
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
outputs = tf.keras.layers.Dense(10, activation="softmax", name="output_layer")(x)

model_2 = tf.keras.Model(inputs, outputs)
model_2.compile(
    loss="categorical_crossentropy",
    optimizer = tf.keras.optimizers.Adam(),
    metrics=["accuracy"]
)

In [ ]:
# Display model architecture
# Should be identical to Model 1
model_2.summary()

### Making a reusable model creation function

In [ ]:
def create_model(input_shape = (224, 224, 3), 
                 output_shape = 10, 
                 learning_rate = 0.001, 
                 training = False):
    """
    Create a model based on EfficientNetV2B0 with built-in data augmentation.
    
    This function encapsulates the model creation logic for easier experimentation.
    By changing the 'training' parameter, we can switch between:
    - Feature extraction (training=False): only train new layers
    - Fine-tuning (training=True): train some base model layers too

    Parameters:
    - input_shape (tuple): Expected shape of input images. Default is (224, 224, 3).
    - output_shape (int): Number of classes for the output layer. Default is 10.
    - learning_rate (float): Learning rate for the Adam optimizer. Default is 0.001.
    - training (bool): Whether the base model is trainable. Default is False (feature extraction).

    Returns:
    - tf.keras.Model: The compiled model with specified input and output settings.
    """

    # Load pre-trained base model
    base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False)
    # Set trainability based on parameter (False for feature extraction, True for fine-tuning)
    base_model.trainable = training

    # Build model architecture
    inputs = tf.keras.layers.Input(shape=input_shape, name="input_layer")
    x = data_augmentation(inputs)  # Augment inputs
    x = base_model(x, training=training)  # Pass through base model
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
    outputs = tf.keras.layers.Dense(output_shape, activation="softmax", name="output_layer")(x)

    # Create and compile model
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        loss="categorical_crossentropy",
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=["accuracy"] )
    
    return model

### Creating a ModelCheckpoint callback

In [ ]:
# ModelCheckpoint allows us to save the best model during training
# This is crucial because:
# 1. Training can be interrupted
# 2. The best model might not be at the final epoch (early stopping concept)
# 3. We can resume training from checkpoints

# Define where to save model weights
# Note: Saving to Colab is temporary - download or save to Google Drive for persistence
checkpoint_path = "ten_percent_model_checkpoints_weights/checkpoint.weights.h5"

# Create ModelCheckpoint callback
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,    # True = save only weights (smaller, faster)
                               # False = save entire model (architecture + weights)
    save_best_only=True,       # Only save when validation performance improves
    save_freq="epoch",         # Check for improvements every epoch
    verbose=1                  # Print message when saving
)

In [ ]:
# Train Model 2 with checkpoint saving
# This trains the model and saves the best weights for later use

initial_epochs = 5  # Store this for later when we do fine-tuning

history_10_percent_data_aug = model_2.fit(
    train_data_10,
    epochs=initial_epochs,
    validation_data=test_data_10,
    validation_steps=int(0.25 * len(test_data_10)),  # Faster validation
    # Use both TensorBoard and ModelCheckpoint callbacks
    callbacks=[
        create_tensorboard_callback("transfer_learning", "10_percent_data_aug"),
        checkpoint_callback  # Save best model weights
    ]
)

### Model 3: a fine-tuned transfer learning model on 10% of the data

In [ ]:
# ============ Model 3: Fine-Tuning on 10% Data ============
# Fine-tuning is the second phase of transfer learning:
# 1. First: train with frozen base (feature extraction) ← Model 2 did this
# 2. Then: unfreeze some base layers and train with lower learning rate ← We do this now

# Create a fresh model with same architecture as Model 2
model_3 = create_model(
    input_shape = (224, 224, 3),
    output_shape = 10,
    learning_rate = 0.001,  # We'll lower this later for fine-tuning
    training = False        # Start frozen, we'll unfreeze specific layers next
) 

In [ ]:
# Load the best weights from Model 2's training
# This gives us a good starting point for fine-tuning
model_3.load_weights(checkpoint_path)

# Evaluate the loaded model to verify it performs as expected
loaded_weights_model_results = model_3.evaluate(test_data_10)
     

In [ ]:
# Inspect all layers in the model to understand the architecture
# This helps us decide which layers to unfreeze for fine-tuning
for layer_number, layer in enumerate(model_3.layers):
  print(f"Layer number: {layer_number+1} | Layer name: {layer.name} | Layer type: {layer} | Trainable? {layer.trainable}")

In [ ]:
# View complete model summary
model_3.summary()

In [ ]:
# ============ Prepare Model 3 for Fine-Tuning ============
# Strategy: Unfreeze only the last few layers of the base model
# Earlier layers learn general features (edges, textures)
# Later layers learn task-specific features (food-specific patterns)

# Access the base model (layer index 2 in the overall architecture)
model_3_base_model = model_3.layers[2]
model_3_base_model.name

# Make the base model trainable
model_3_base_model.trainable = True

# Freeze all layers except the last 10
# This is selective fine-tuning - we only train the most task-specific layers
for layer in model_3_base_model.layers[:-10]:
    layer.trainable = False

# Recompile with a lower learning rate
# Lower LR (0.0001 vs 0.001) prevents destroying pre-trained weights
model_3.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # 10x lower than initial training
    metrics=["accuracy"]
)

In [ ]:
# Optional: Check which specific base model layers are now trainable
# Uncomment to see the last 10 layers that will be fine-tuned
# for layer_number, layer in enumerate(model_3_base_model.layers):
#   print(layer_number, layer.name, layer.trainable)

In [ ]:
# Count total trainable variables
# This should be higher than Model 2 since we unfroze some base layers
print(len(model_2.trainable_variables))

In [ ]:
# ============ Fine-Tune Model 3 ============
# Continue training for 5 more epochs with unfrozen layers

# Calculate total epochs (initial training + fine-tuning)
fine_tune_epochs = initial_epochs + 5  # 5 + 5 = 10 total epochs

# Resume training from where Model 2 left off
history_fine_10_percent_data_aug = model_3.fit(
    train_data_10,
    epochs=fine_tune_epochs,                                    # Train until epoch 10
    validation_data=test_data_10,
    initial_epoch=history_10_percent_data_aug.epoch[-1],       # Start from epoch 5 (where Model 2 ended)
    validation_steps=int(0.25 * len(test_data_10)),
    callbacks=[create_tensorboard_callback("transfer_learning", "10_percent_fine_tune_last_10")]
) 

In [ ]:
# Evaluate fine-tuned model on test data
# Compare this to Model 2's results to see if fine-tuning helped
results_fine_tune_10_percent = model_3.evaluate(test_data_10)

In [ ]:
# ============ Utility Function for Comparing Training Phases ============
def compare_historys(original_history, new_history, initial_epochs=5):
    """
    Compare two training phases (e.g., feature extraction vs fine-tuning).
    
    This function combines metrics from both phases and plots them together,
    with a vertical line showing where fine-tuning began.
    
    Args:
        original_history: History from initial training (feature extraction)
        new_history: History from fine-tuning
        initial_epochs: Number of epochs in original training (for vertical line)
    """
    # Extract metrics from original training phase
    acc = original_history.history["accuracy"]
    loss = original_history.history["loss"]
    val_acc = original_history.history["val_accuracy"]
    val_loss = original_history.history["val_loss"]

    print(len(acc))  # Debug: show number of original epochs

    # Combine original and new history
    # This creates a continuous timeline across both training phases
    total_acc = acc + new_history.history["accuracy"]
    total_loss = loss + new_history.history["loss"]
    total_val_acc = val_acc + new_history.history["val_accuracy"]
    total_val_loss = val_loss + new_history.history["val_loss"]

    print(len(total_acc))  # Debug: show total epochs
    print(total_acc)       # Debug: show all accuracy values

    # Create comparison plots
    plt.figure(figsize=(8, 8))
    
    # Plot accuracy
    plt.subplot(2, 1, 1)
    plt.plot(total_acc, label='Training Accuracy')
    plt.plot(total_val_acc, label='Validation Accuracy')
    # Add vertical line showing where fine-tuning started
    plt.plot([initial_epochs-1, initial_epochs-1],
              plt.ylim(), label='Start Fine Tuning')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')

    # Plot loss
    plt.subplot(2, 1, 2)
    plt.plot(total_loss, label='Training Loss')
    plt.plot(total_val_loss, label='Validation Loss')
    plt.plot([initial_epochs-1, initial_epochs-1],
              plt.ylim(), label='Start Fine Tuning')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('epoch')
    plt.show()
     

In [ ]:
# Visualize the complete training timeline
# This shows both feature extraction (epochs 0-4) and fine-tuning (epochs 5-9)
compare_historys(original_history=history_model_2,
                 new_history=history_fine_10_percent_data_aug,
                 initial_epochs=5)

## Model 4: a fine-tuned transfer learning model on 100% of the data

### All available data loading

In [ ]:
# ============ Load Full Dataset (100% of data) ============
# This is the complete dataset - much more training data
# Should lead to better model performance than 10% or 1% versions

# Define paths to full dataset
train_dir = "10_food_classes_all_data/train/"
test_dir = "10_food_classes_all_data/test/"

IMG_SIZE = (224, 224)

# Create training dataset from all available data
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size = IMG_SIZE,
    label_mode = "categorical",
    batch_size = 32
)

# Create test dataset
test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size = IMG_SIZE,
    label_mode = "categorical"
)

# Verify class names match previous datasets
train_data.class_names

In [ ]:
# Quick test: Evaluate Model 3 (trained on 10% data) on the full test set
# This shows performance when a model trained on limited data meets more test examples
model_3.evaluate(test_data)

In [ ]:
# ============ Model 4: Fine-Tuning on 100% Data ============
# Create a new model for training on the full dataset
model_4 = create_model(
    input_shape = (224, 224, 3),
    output_shape = 10,
    learning_rate = 0.001,
    training = False  # Start with frozen base, will unfreeze for fine-tuning
) 

In [ ]:
# Load the checkpoint weights from 10% data training
# This gives us a head start instead of training from scratch
model_4.load_weights(checkpoint_path)
loaded_weights_model_results = model_4.evaluate(test_data)

In [ ]:
# ============ Prepare Model 4 for Fine-Tuning ============
# Same fine-tuning strategy as Model 3: unfreeze last 10 layers

# Access the base model (EfficientNetV2B0)
model_4_base_model = model_4.layers[2]
model_4_base_model.name

# Enable training for base model
model_4_base_model.trainable = True

# Freeze all but the last 10 layers
for layer in model_4_base_model.layers[:-10]:
    layer.trainable = False

# Recompile with lower learning rate for fine-tuning
model_4.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # 10x lower LR
    metrics=["accuracy"]
)

In [ ]:
# ============ Fine-Tune Model 4 on Full Dataset ============
# This is the final model - trained on all available data with fine-tuning

# Continue for 5 more epochs beyond the initial training
fine_tune_epochs = initial_epochs + 5

# Train on full dataset
history_fine_all_data_aug = model_4.fit(
    train_data,                                                # Full training data
    epochs=fine_tune_epochs,
    validation_data=test_data,
    initial_epoch=history_10_percent_data_aug.epoch[-1],      # Continue from epoch 5
    validation_steps=int(0.25 * len(test_data)),
    callbacks=[create_tensorboard_callback("transfer_learning", "full_10_classes_fine_tune_last_10")]
) 

In [ ]:
# Compare performance: 10% data vs 100% data fine-tuning
# This shows the impact of having more training data
compare_historys(original_history=history_10_percent_data_aug,
                 new_history=history_fine_all_data_aug,
                 initial_epochs=5)